In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODELS_DIR = PROJECT_DIR / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data exists:", RAW_DIR.exists())
print("Processed data exists:", PROCESSED_DIR.exists())
print("Models directory:", MODELS_DIR)

Project: /Users/sejal/Desktop/home-credit-default-risk-intelligence
Raw data exists: True
Processed data exists: True
Models directory: /Users/sejal/Desktop/home-credit-default-risk-intelligence/models


In [2]:
feature_files = {
    "previous_features": "previous_application_features.parquet",
    "bureau_features": "bureau_features.parquet",
    "installment_features": "installment_features.parquet",
    "pos_features": "pos_cash_features.parquet",
    "credit_card_features": "credit_card_features.parquet",
}

features = {}

for name, filename in feature_files.items():
    path = PROCESSED_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Missing feature file: {path}")

    features[name] = pd.read_parquet(path)

    print(
        f"{name}: {features[name].shape} | "
        f"unique applicants: "
        f"{features[name]['SK_ID_CURR'].nunique():,}"
    )

previous_features: (338857, 17) | unique applicants: 338,857
bureau_features: (305811, 23) | unique applicants: 305,811
installment_features: (339587, 15) | unique applicants: 339,587
pos_features: (337252, 14) | unique applicants: 337,252
credit_card_features: (103558, 19) | unique applicants: 103,558


In [3]:
train = pd.read_csv(RAW_DIR / "application_train.csv")
test = pd.read_csv(RAW_DIR / "application_test.csv")

print("Training data:", train.shape)
print("Test data:", test.shape)
print("\nTarget distribution:")
print(train["TARGET"].value_counts())
print("\nTarget proportions:")
print(train["TARGET"].value_counts(normalize=True).round(4))

Training data: (307511, 122)
Test data: (48744, 121)

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Target proportions:
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64


In [4]:
# Merge all saved historical features with application data

train_model = train.merge(
    features["previous_features"],
    on="SK_ID_CURR",
    how="left"
)

test_model = test.merge(
    features["previous_features"],
    on="SK_ID_CURR",
    how="left"
)

for name in ["bureau_features", "installment_features",
             "pos_features", "credit_card_features"]:

    train_model = train_model.merge(
        features[name],
        on="SK_ID_CURR",
        how="left"
    )

    test_model = test_model.merge(
        features[name],
        on="SK_ID_CURR",
        how="left"
    )

print("Train E6:", train_model.shape)
print("Test E6:", test_model.shape)

print("\nDuplicate training IDs:", train_model["SK_ID_CURR"].duplicated().sum())
print("Duplicate test IDs:", test_model["SK_ID_CURR"].duplicated().sum())

assert train_model["SK_ID_CURR"].is_unique
assert test_model["SK_ID_CURR"].is_unique
assert "TARGET" in train_model.columns
assert "TARGET" not in test_model.columns

print("\nE6 datasets reconstructed successfully!")

Train E6: (307511, 205)
Test E6: (48744, 204)

Duplicate training IDs: 0
Duplicate test IDs: 0

E6 datasets reconstructed successfully!


In [5]:
def add_engineered_features(df):
    df = df.copy()

    df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365.25).clip(lower=0)
    df["EMPLOYED_YEARS"] = (
        -df["DAYS_EMPLOYED"].where(df["DAYS_EMPLOYED"] < 0) / 365.25
    )

    df["EMPLOYMENT_AGE_RATIO"] = (
        df["EMPLOYED_YEARS"] / df["AGE_YEARS"].replace(0, np.nan)
    )

    df["CREDIT_INCOME_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )
    df["ANNUITY_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )
    df["CREDIT_ANNUITY_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_ANNUITY"].replace(0, np.nan)
    )
    df["GOODS_CREDIT_RATIO"] = (
        df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"].replace(0, np.nan)
    )
    df["INCOME_CREDIT_RATIO"] = (
        df["AMT_INCOME_TOTAL"] / df["AMT_CREDIT"].replace(0, np.nan)
    )

    family_size = df["CNT_FAM_MEMBERS"].replace(0, np.nan)

    df["INCOME_PER_FAMILY_MEMBER"] = df["AMT_INCOME_TOTAL"] / family_size
    df["CREDIT_PER_FAMILY_MEMBER"] = df["AMT_CREDIT"] / family_size
    df["ANNUITY_PER_FAMILY_MEMBER"] = df["AMT_ANNUITY"] / family_size

    children = df["CNT_CHILDREN"].replace(0, np.nan)
    df["INCOME_PER_CHILD"] = df["AMT_INCOME_TOTAL"] / children

    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)
    df["EXT_SOURCE_PRODUCT"] = df[ext_cols].prod(axis=1, min_count=1)

    df["EXT_SOURCE_WEIGHTED"] = (
        0.2 * df["EXT_SOURCE_1"]
        + 0.3 * df["EXT_SOURCE_2"]
        + 0.5 * df["EXT_SOURCE_3"]
    )

    df["CREDIT_STRESS_EXT"] = (
        df["CREDIT_INCOME_RATIO"] / (df["EXT_SOURCE_MEAN"] + 0.1)
    )
    df["ANNUITY_STRESS_EXT"] = (
        df["ANNUITY_INCOME_RATIO"] / (df["EXT_SOURCE_MEAN"] + 0.1)
    )

    return df


train_fe = add_engineered_features(train_model)
test_fe = add_engineered_features(test_model)

print("Train FE:", train_fe.shape)
print("Test FE:", test_fe.shape)
print("Feature count:", len(train_fe.columns) - 2)

Train FE: (307511, 225)
Test FE: (48744, 224)
Feature count: 223


In [6]:
from sklearn.model_selection import train_test_split

TARGET = "TARGET"
ID = "SK_ID_CURR"

X = train_fe.drop(columns=[TARGET, ID])
y = train_fe[TARGET]

X_train_fe, X_valid_fe, y_train_fe, y_valid_fe = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features_fe = X_train_fe.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Training rows:", len(X_train_fe))
print("Validation rows:", len(X_valid_fe))
print("Features:", X_train_fe.shape[1])
print("Categorical features:", len(categorical_features_fe))

Training rows: 246008
Validation rows: 61503
Features: 223
Categorical features: 16


/var/folders/jj/1fp60rtn35g1z319tfv84jcr0000gn/T/ipykernel_9775/547193330.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_fe = X_train_fe.select_dtypes(


In [8]:
# Reconstruct E6 using the saved feature tables

train_model_e6 = train_model.copy()
test_model_e6 = test_model.copy()

for name in [
    "bureau_features",
    "installment_features",
    "pos_features",
    "credit_card_features"
]:
    train_model_e6 = train_model_e6.merge(
        features[name],
        on="SK_ID_CURR",
        how="left"
    )

    test_model_e6 = test_model_e6.merge(
        features[name],
        on="SK_ID_CURR",
        how="left"
    )

print("E6 train:", train_model_e6.shape)
print("E6 test:", test_model_e6.shape)

print(
    "Features excluding ID and TARGET:",
    train_model_e6.drop(columns=["SK_ID_CURR", "TARGET"]).shape[1]
)

assert train_model_e6["SK_ID_CURR"].is_unique
assert test_model_e6["SK_ID_CURR"].is_unique

E6 train: (307511, 272)
E6 test: (48744, 271)
Features excluding ID and TARGET: 270


In [9]:
print("Feature-engineered columns:", train_fe.shape[1])
print("Features excluding ID and TARGET:", X_train_fe.shape[1])

Feature-engineered columns: 225
Features excluding ID and TARGET: 223


In [10]:
# Rebuild E6 cleanly — each historical table merged once

train_model_e6 = train.copy()
test_model_e6 = test.copy()

for name in [
    "previous_features",
    "bureau_features",
    "installment_features",
    "pos_features",
    "credit_card_features",
]:
    train_model_e6 = train_model_e6.merge(
        features[name],
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

    test_model_e6 = test_model_e6.merge(
        features[name],
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

print("E6 train:", train_model_e6.shape)
print("E6 test:", test_model_e6.shape)

print(
    "E6 features:",
    train_model_e6.drop(columns=["SK_ID_CURR", "TARGET"]).shape[1]
)

assert train_model_e6["SK_ID_CURR"].is_unique
assert test_model_e6["SK_ID_CURR"].is_unique

print("E6 rebuilt successfully.")

E6 train: (307511, 205)
E6 test: (48744, 204)
E6 features: 203
E6 rebuilt successfully.


In [11]:
train_fe = add_engineered_features(train_model_e6)
test_fe = add_engineered_features(test_model_e6)

X = train_fe.drop(columns=["SK_ID_CURR", "TARGET"])
y = train_fe["TARGET"]

X_train_fe, X_valid_fe, y_train_fe, y_valid_fe = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

categorical_features_fe = X_train_fe.select_dtypes(
    include=["object", "category", "str"]
).columns.tolist()

print("Train FE:", train_fe.shape)
print("Test FE:", test_fe.shape)
print("Features:", X_train_fe.shape[1])
print("Categorical features:", len(categorical_features_fe))

Train FE: (307511, 225)
Test FE: (48744, 224)
Features: 223
Categorical features: 16


In [12]:
from catboost import CatBoostClassifier
import numpy as np
import joblib

X_train_cat = X_train_fe.copy()
X_valid_cat = X_valid_fe.copy()

for col in categorical_features_fe:
    X_train_cat[col] = (
        X_train_cat[col]
        .astype("str")
        .replace("nan", "MISSING")
    )
    X_valid_cat[col] = (
        X_valid_cat[col]
        .astype("str")
        .replace("nan", "MISSING")
    )

cat_features_idx = [
    X_train_cat.columns.get_loc(col)
    for col in categorical_features_fe
]

print("Training shape:", X_train_cat.shape)
print("Categorical features:", len(cat_features_idx))

Training shape: (246008, 223)
Categorical features: 16


In [13]:
cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.035,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=42,
    thread_count=-1,
    verbose=250,
    allow_writing_files=False
)

cat_model.fit(
    X_train_cat,
    y_train_fe,
    cat_features=cat_features_idx,
    eval_set=(X_valid_cat, y_valid_fe),
    early_stopping_rounds=100,
    verbose=250
)

CatBoostError: Invalid type for cat_feature[non-default value idx=187,feature_idx=9]=nan : cat_features must be integer or string, real number values and NaN values should be converted to string.

In [14]:
# Correctly convert all categorical values to strings
# and replace missing values before CatBoost training.

for col in categorical_features_fe:
    X_train_cat[col] = (
        X_train_fe[col]
        .astype("object")
        .where(X_train_fe[col].notna(), "MISSING")
        .astype(str)
    )

    X_valid_cat[col] = (
        X_valid_fe[col]
        .astype("object")
        .where(X_valid_fe[col].notna(), "MISSING")
        .astype(str)
    )

# Verify that no categorical column contains missing values
train_missing = X_train_cat[categorical_features_fe].isna().sum().sum()
valid_missing = X_valid_cat[categorical_features_fe].isna().sum().sum()

print("Missing categorical values in training:", train_missing)
print("Missing categorical values in validation:", valid_missing)

assert train_missing == 0
assert valid_missing == 0

cat_features_idx = [
    X_train_cat.columns.get_loc(col)
    for col in categorical_features_fe
]

print("Categorical preprocessing fixed.")

Missing categorical values in training: 0
Missing categorical values in validation: 0
Categorical preprocessing fixed.


In [15]:
cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.035,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=5,
    random_seed=42,
    thread_count=-1,
    verbose=250,
    allow_writing_files=False
)

cat_model.fit(
    X_train_cat,
    y_train_fe,
    cat_features=cat_features_idx,
    eval_set=(X_valid_cat, y_valid_fe),
    early_stopping_rounds=100,
    verbose=250
)

0:	test: 0.6351611	best: 0.6351611 (0)	total: 338ms	remaining: 8m 26s
250:	test: 0.7789174	best: 0.7789174 (250)	total: 47.6s	remaining: 3m 57s
500:	test: 0.7840606	best: 0.7840664 (498)	total: 1m 39s	remaining: 3m 17s
750:	test: 0.7860755	best: 0.7860755 (750)	total: 2m 33s	remaining: 2m 33s
1000:	test: 0.7871137	best: 0.7871138 (996)	total: 3m 28s	remaining: 1m 43s
1250:	test: 0.7879707	best: 0.7879707 (1250)	total: 4m 23s	remaining: 52.5s
1499:	test: 0.7885059	best: 0.7885207 (1490)	total: 5m 19s	remaining: 0us

bestTest = 0.788520667
bestIteration = 1490

Shrink model to first 1491 iterations.


CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=1500, l2_leaf_reg=5, learning_rate=0.035, loss_function='Logloss', random_seed=42, verbose=250)

In [16]:
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib

# Evaluate validation predictions
valid_prob_cat = cat_model.predict_proba(X_valid_cat)[:, 1]

roc_auc = roc_auc_score(y_valid_fe, valid_prob_cat)
pr_auc = average_precision_score(y_valid_fe, valid_prob_cat)

print(f"CatBoost ROC-AUC: {roc_auc:.4f}")
print(f"CatBoost PR-AUC : {pr_auc:.4f}")

# Save trained model
MODELS_DIR.mkdir(parents=True, exist_ok=True)

cat_model.save_model(str(MODELS_DIR / "catboost_model.cbm"))

# Save the feature schema and validation predictions
joblib.dump(
    {
        "feature_columns": X_train_cat.columns.tolist(),
        "categorical_features": categorical_features_fe,
        "risk_band_thresholds": [0.05, 0.10, 0.20],
    },
    MODELS_DIR / "model_metadata.joblib"
)

pd.DataFrame({
    "SK_ID_CURR": train_fe.loc[X_valid_fe.index, "SK_ID_CURR"].values,
    "actual_default": y_valid_fe.values,
    "predicted_risk": valid_prob_cat
}).to_parquet(
    PROCESSED_DIR / "validation_predictions.parquet",
    index=False
)

print("\nModel saved:", MODELS_DIR / "catboost_model.cbm")
print("Metadata and validation predictions saved.")

CatBoost ROC-AUC: 0.7885
CatBoost PR-AUC : 0.2893

Model saved: /Users/sejal/Desktop/home-credit-default-risk-intelligence/models/catboost_model.cbm
Metadata and validation predictions saved.


In [17]:
for path in sorted(MODELS_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name}: {path.stat().st_size / (1024**2):.2f} MB")

catboost_model.cbm: 3.39 MB
model_metadata.joblib: 0.00 MB


In [18]:
# Save validation applicants for the interactive app
feature_store = X_valid_cat.copy()

feature_store.insert(
    0,
    "SK_ID_CURR",
    train_fe.loc[X_valid_cat.index, "SK_ID_CURR"].values
)

feature_store.to_parquet(
    PROCESSED_DIR / "applicant_feature_store.parquet",
    index=False
)

print("Applicant feature store saved.")
print("Shape:", feature_store.shape)
print("Unique applicants:", feature_store["SK_ID_CURR"].nunique())

Applicant feature store saved.
Shape: (61503, 224)
Unique applicants: 61503


In [19]:
import sys
print(sys.executable)
print(sys.version)


/opt/anaconda3/bin/python3
3.14.6 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:30:20) [Clang 20.1.8 ]
